# SFVP AI Clone — Notebook 04: Enhance + Export
**Tools:** CodeFormer (face restoration) + FFmpeg (social media encoding)

Cleans up AI artifacts, sharpens the face, and exports in the right format for Instagram Reels, TikTok, or YouTube.

**Input:** video from Notebook 02 or 03 (in Drive/SFVP_Clone/outputs/)
**Output:** polished, post-ready MP4

**Runtime:** T4 GPU

In [ ]:
# CELL 1 — Install CodeFormer (run once per session)
import os

if not os.path.exists('/content/CodeFormer'):
    !git clone https://github.com/sczhou/CodeFormer.git /content/CodeFormer -q

%cd /content/CodeFormer
!pip install -r requirements.txt -q
!apt-get install -y ffmpeg -q

# Download pretrained models
!python scripts/download_pretrained_models.py facelib -q
!python scripts/download_pretrained_models.py CodeFormer -q

print('CodeFormer ready.')

In [ ]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'

# List videos in outputs folder
outputs_dir = f'{DRIVE_BASE}/outputs'
vids = [f for f in os.listdir(outputs_dir) if f.endswith('.mp4') and 'final' not in f]
print('Available videos to enhance:')
for i, v in enumerate(vids):
    print(f'  [{i}] {v}')

In [ ]:
# CELL 3 — CONFIGURE

# Input video (from your outputs/ folder)
INPUT_VIDEO = 'lipsync_output_01.mp4'  # <-- CHANGE THIS

# Output filename (saved back to outputs/ with FINAL_ prefix)
OUTPUT_VIDEO = 'FINAL_ad_01.mp4'  # <-- CHANGE THIS

# CodeFormer fidelity weight:
# 0.0 = maximum restoration (may change face slightly)
# 1.0 = maximum fidelity to original (preserves identity)
# 0.7 is a good starting point — restores quality while keeping Shennel's likeness
FIDELITY_WEIGHT = 0.7

# Social media export format
# 'reels'   = 9:16 vertical (Instagram Reels, TikTok, YouTube Shorts)
# 'square'  = 1:1 (Instagram feed)
# 'wide'    = 16:9 (YouTube, Facebook)
# 'original'= keep original size
EXPORT_FORMAT = 'reels'  # <-- CHANGE THIS

print(f'Input:  {INPUT_VIDEO}')
print(f'Output: {OUTPUT_VIDEO}')
print(f'Fidelity: {FIDELITY_WEIGHT} | Format: {EXPORT_FORMAT}')

In [ ]:
# CELL 4 — Run CodeFormer face enhancement on each frame
import subprocess
import shutil

%cd /content/CodeFormer

src = f'{DRIVE_BASE}/outputs/{INPUT_VIDEO}'
local_input = '/content/input_video.mp4'
shutil.copy(src, local_input)

enhanced_dir = '/content/cf_output'
os.makedirs(enhanced_dir, exist_ok=True)

cmd = [
    'python', 'inference_codeformer.py',
    '-w', str(FIDELITY_WEIGHT),
    '--input_path', local_input,
    '--output_path', enhanced_dir,
    '--face_upsample',  # upsample face region
    '--bg_upsampler', 'realesrgan',  # upscale background too
    '--upscale', '2',
]

print('Running CodeFormer enhancement (this takes a few minutes)...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('ERROR:', result.stderr[-2000:])
else:
    print('Enhancement complete!')
    # Find enhanced video
    import glob
    enhanced = glob.glob(f'{enhanced_dir}/video/*.mp4')
    if enhanced:
        shutil.copy(enhanced[0], '/content/enhanced.mp4')
        print('Enhanced video ready.')
    else:
        print('WARNING: No enhanced video found — check error output')
        print(result.stdout[-2000:])

In [ ]:
# CELL 5 — Export in social media format

FORMAT_SETTINGS = {
    'reels':    {'scale': 'scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920', 'crf': '18'},
    'square':   {'scale': 'scale=1080:1080:force_original_aspect_ratio=increase,crop=1080:1080', 'crf': '18'},
    'wide':     {'scale': 'scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080', 'crf': '18'},
    'original': {'scale': 'scale=trunc(iw/2)*2:trunc(ih/2)*2', 'crf': '18'},
}

settings = FORMAT_SETTINGS.get(EXPORT_FORMAT, FORMAT_SETTINGS['original'])
local_final = '/content/final_output.mp4'

subprocess.run([
    'ffmpeg', '-y', '-i', '/content/enhanced.mp4',
    '-vf', settings['scale'],
    '-c:v', 'libx264', '-crf', settings['crf'],
    '-preset', 'slow',
    '-c:a', 'aac', '-b:a', '192k',
    '-movflags', '+faststart',  # web-optimized
    local_final
], capture_output=True)

# Save to Drive
drive_final = f'{DRIVE_BASE}/outputs/{OUTPUT_VIDEO}'
shutil.copy(local_final, drive_final)
print(f'Exported as {EXPORT_FORMAT.upper()} format.')
print(f'Saved to: {drive_final}')

In [ ]:
# CELL 6 — Preview final output
from IPython.display import HTML
from base64 import b64encode

with open(local_final, 'rb') as f:
    vb64 = b64encode(f.read()).decode()

HTML(f'''
<video width="360" controls>
  <source src="data:video/mp4;base64,{vb64}" type="video/mp4">
</video>
<p><b>This is your post-ready video.</b><br>
It's saved to your Google Drive at: outputs/{OUTPUT_VIDEO}<br><br>
If the face looks over-smoothed, lower FIDELITY_WEIGHT toward 0.9<br>
If you still see artifacts, lower it toward 0.5</p>
''')

In [ ]:
# CELL 7 — (OPTIONAL) Add text overlay / lower thirds
# Uncomment and customize to add your name/CTA as a text overlay

# OVERLAY_TEXT = 'Sactown\'s Finest Vinyl & Print'
# CTA_TEXT = '@sactownfinest | DM to order'
# 
# local_with_text = '/content/final_with_text.mp4'
# subprocess.run([
#     'ffmpeg', '-y', '-i', local_final,
#     '-vf', (
#         f"drawtext=text='{OVERLAY_TEXT}':fontsize=48:fontcolor=white:"
#         f"x=(w-text_w)/2:y=h-200:box=1:boxcolor=black@0.6:boxborderw=10,"
#         f"drawtext=text='{CTA_TEXT}':fontsize=32:fontcolor=#E8FF00:"
#         f"x=(w-text_w)/2:y=h-130:box=1:boxcolor=black@0.6:boxborderw=8"
#     ),
#     '-c:v', 'libx264', '-crf', '18', '-c:a', 'copy',
#     local_with_text
# ], capture_output=True)
# shutil.copy(local_with_text, f'{DRIVE_BASE}/outputs/FINAL_with_text_{OUTPUT_VIDEO}')
# print('Version with text overlay saved.')